# Notebook #1 — v0-baseline (unblocks P0-7)

This notebook is the **C1 cloud trainer**: it pulls the `convaiinnovations/laya` checkpoints from Hugging Face and runs the **C2 eval environment** (the `layaenv` package + frozen grid that live in this repo) against them.

Outputs, handed back for local validation and gating:
- `report.jsonl` — frozen-schema metric rows (one per cell x model variant)
- `probe.jsonl` — raw SDK output on a tiny example, so the adapter can be tightened against the real API
- `laya-artifacts.zip` — the three files, zipped for download

Requirements: Accelerator `GPU T4 x2`, Internet `On`. Edit the CONFIG cell, then Run All (~15–30 min).

In [ ]:
# ============ CONFIG ============
# C2 eval environment is cloned from this repo (public). Pin REF to a
# commit when a gate needs to re-check the exact code that ran.
REPO_URL = "https://github.com/ahmed-alqasaby/laya.git"
REF = "main"
REPO_DIR = "/kaggle/working/laya"

GRID = "configs/eval_grid.yaml"
OUTDIR = "/kaggle/working"

# variant -> (Hugging Face repo, subfolder). Variant names are frozen
# in the grid config.
MODELS = {
    "english":         ("convaiinnovations/laya", None),
    "multilingual":    ("convaiinnovations/laya", "multilingual"),
    "typed_decisions": ("convaiinnovations/laya", "typed-decisions"),
    "router":          ("router", None),
}
VARIANTS = ["english", "multilingual", "typed_decisions", "router"]


In [ ]:
import os, subprocess, sys
os.environ["USE_TF"] = "0"  # transformers + TF can deadlock; see laya model card

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "laya", "datasets", "scipy", "pyyaml", "psutil"], check=True)

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", REF, REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, REPO_DIR)
if not os.path.isdir(os.path.join(REPO_DIR, "layaenv")):
    raise RuntimeError(
        f"C2 package layaenv missing in {REPO_DIR} — is the C2 work pushed"
        f" to {REF} on {REPO_URL}? Re-run after the push, then Run All again."
    )
if not os.path.isfile(os.path.join(REPO_DIR, GRID)):
    raise RuntimeError(f"grid config {GRID} missing in {REPO_DIR} — repo clone is stale")


import layaenv
print("C2 eval environment imported from", os.path.dirname(layaenv.__file__))


In [ ]:
# Pin the real SDK contract before running the grid: one tiny call per
# variant, raw result dumped to probe.jsonl.
import json, time

state = {"body": "Duplicate charge on invoice #4411"}
questions = {
    "department": {
        "type": "choice",
        "instructions": "Which department should handle this request?",
        "criteria": {
            "billing": "invoices, payments, refunds",
            "technical": "bugs, outages, errors",
            "other": "everything else",
        },
    },
    "urgent": {"type": "noul", "instructions": "Is this urgent?"},
    "priority": {"type": "score", "instructions": "Rate the priority.", "criteria": ["low", "medium", "high"]},
}

probe = []
for variant in VARIANTS:
    repo, sub = MODELS[variant]
    backend = layaenv.backends.load_backend(variant, repo, sub)
    t0 = time.perf_counter()
    try:
        res = backend.predict(state, questions)
        ok = True
    except Exception as exc:
        res = {"error": str(exc)}
        ok = False
    dt = (time.perf_counter() - t0) * 1000.0
    probe.append({"variant": variant, "ok": ok, "elapsed_ms": round(dt, 1), "result": res})
    print(variant, "ok" if ok else "FAIL", f"{dt:.1f} ms")

with open(os.path.join(OUTDIR, "probe.jsonl"), "w") as f:
    for row in probe:
        f.write(json.dumps(row, default=str) + "\n")
print("probe written ->", os.path.join(OUTDIR, "probe.jsonl"))


In [ ]:
# Run the frozen grid with the real checkpoints. Cells whose eval data
# is not reachable are skipped and listed, never silently dropped.
rows, skipped = layaenv.run_grid(
    os.path.join(REPO_DIR, GRID),
    repo_root=REPO_DIR,
    outdir=OUTDIR,
    model_registry=MODELS,
    variants=VARIANTS,
)
for s in skipped:
    print("SKIP:", s)
print(f"produced {len(rows)} report rows")


In [ ]:
# Wrap up: frozen-schema report, summary table, self-describing zip.
import zipfile

zip_path = layaenv.write_artifacts(OUTDIR, rows, skipped)
print(layaenv.schema.render_metric_table(rows))
with zipfile.ZipFile(zip_path) as z:
    print("artifact contents:", ", ".join(z.namelist()))
print("DONE:", zip_path)
